# Multi-tool Agent with Text2Cypher

You will modify the agent to add a _Text to Cypher_ retriever tool.

The Text to Cypher tool will allow the agent to create queries to retrieve more specific information such as facts and figures.

---

## Setup

Ensure you have completed Lab 1 and filled in your credentials in `CONFIG.txt`.

## 1. Configuration

In [ ]:
# Load configuration from CONFIG.txt
from dotenv import load_dotenv
import os

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

# Validate configuration
errors = []
if not OPENAI_API_KEY or "your-" in OPENAI_API_KEY or "sk-your" in OPENAI_API_KEY:
    errors.append("Set OPENAI_API_KEY in CONFIG.txt")
if not NEO4J_URI or "xxxxxxxx" in NEO4J_URI:
    errors.append("Set NEO4J_URI in CONFIG.txt")
if not NEO4J_PASSWORD or "your_password" in NEO4J_PASSWORD:
    errors.append("Set NEO4J_PASSWORD in CONFIG.txt")

if errors:
    print("ERROR: Configuration incomplete!")
    for e in errors:
        print(f"  - {e}")
else:
    print(f"Model: {MODEL_ID}")
    print(f"Region: {REGION}")
    print(f"Neo4j URI: {NEO4J_URI}")
    print("\nConfiguration OK!")

## 2. Setup

Load the environment variables, import the required Python modules, and set up the base tools.

In [ ]:
# Install missing packages
%pip install langgraph langchain-aws neo4j-graphrag boto3 openai -q

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import VectorCypherRetriever, Text2CypherRetriever
from neo4j_graphrag.schema import get_schema
from neo4j_graphrag.llm import BedrockLLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports successful!")

In [ ]:
# Connect to Neo4j and create embedder
driver = GraphDatabase.driver(
    NEO4J_URI, 
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)
driver.verify_connectivity()

# Create the embedding model using OpenAI (matching the vector index)
embedder = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY,
    model="text-embedding-ada-002",
)

print("Connected to Neo4j and created embedder!")

## 3. Create Vector Retriever

Define the retrieval query for vector search with graph context.

In [ ]:
# Retrieval query for vector search with graph context
retrieval_query = """
MATCH (node)-[:FROM_DOCUMENT]-(doc:Document)-[:FILED]-(company:Company)
OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
WITH node, score, company, collect(risk.name)[0..20] AS risks
WHERE score IS NOT NULL
RETURN 
    node.text AS text,
    score,
    {company: company.name, risks: risks} AS metadata
ORDER BY score DESC
"""

# Create vector retriever
vector_retriever = VectorCypherRetriever(
    driver=driver,
    index_name="chunkEmbeddings",
    embedder=embedder,
    retrieval_query=retrieval_query,
)

print("Vector retriever created!")

## 4. Create Text2Cypher Retriever

The Text to Cypher tool uses a separate LLM to generate the Cypher. This is useful as different models and settings are more effective at generating Cypher.

Create a `cypher_llm` using AWS Bedrock.

In [ ]:
# Create a separate LLM for Cypher generation using AWS Bedrock
cypher_llm = BedrockLLM(
    model_id=MODEL_ID,
    region_name=REGION,
)

print("Cypher LLM created!")

The Text to Cypher tool requires a prompt which instructs the LLM on how to generate the Cypher.

Create a `cypher_prompt` which accepts the graph `schema` and the user's `question`.

In [ ]:
# Create a cypher generation prompt with modern Cypher best practices
cypher_prompt = """Task: Generate a Cypher statement to query a graph database.

Instructions:
- Use only the provided relationship types and properties in the schema.
- Do not use any other relationship types or properties that are not provided.
- Only filter by name when a specific entity name is mentioned in the question.
  When filtering by name, use case-insensitive matching:
  `WHERE toLower(node.name) CONTAINS toLower('ActualEntityName')`
- Do NOT add name filters if no specific entity name is mentioned in the question.

Modern Cypher Requirements:
- Use `elementId(node)` instead of `id(node)` (id() is removed in Neo4j 5+).
- Use `count{{pattern}}` instead of `size((pattern))` for counting patterns.
- Use `EXISTS {{MATCH pattern}}` instead of `exists((pattern))` for existence checks.
- When using ORDER BY, filter NULL values first: `WHERE property IS NOT NULL ORDER BY property`.
- Use explicit grouping with WITH clauses for aggregations.
- Limit collected results when appropriate: `collect(item)[0..20]`.

Schema:
{schema}

Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

The question is:
{query_text}"""

print("Cypher prompt defined!")

The prompt can include specific instructions on how to generate Cypher, for example, this instruction:

> Use `WHERE toLower(node.name) CONTAINS toLower('name')` to filter nodes by name.

... tells the LLM to use case insensitive and wild card matching when searching by company name.

---

Create the Text2Cypher retriever using the schema and custom prompt.

In [ ]:
# Create the Text2Cypher retriever
text2cypher_retriever = Text2CypherRetriever(
    driver=driver,
    llm=cypher_llm,
    neo4j_schema=get_schema(driver),
    custom_prompt=cypher_prompt,
)

print("Text2Cypher retriever created!")

**Important Note on Text2Cypher:**

You are trusting the generation of Cypher to the LLM. It may generate invalid Cypher queries that could corrupt data in the graph or provide access to sensitive information.

In a production environment, you should ensure that access to data is limited, and sufficient security is in place to prevent malicious queries.

---

## 5. Define Tools

Create the tools for the agent. The multi-tool agent will have three tools:

1. `get_graph_schema` - Get the database schema
2. `retrieve_financial_documents` - Search documents semantically
3. `query_database` - Query specific facts from the database

In [ ]:
@tool
def get_graph_schema() -> str:
    """Get the schema of the graph database including node labels, relationships, and properties."""
    return get_schema(driver)


@tool
def retrieve_financial_documents(query: str) -> str:
    """Find details about companies in their financial documents using semantic search.
    
    Args:
        query: The search query to find relevant documents
    """
    try:
        results = vector_retriever.search(query_text=query, top_k=3)
        if not results.items:
            return "No documents found matching the query."
        return "\n\n".join(item.content for item in results.items)
    except Exception as e:
        return f"Error searching documents: {e}"


@tool
def query_database(query: str) -> str:
    """Get answers to specific questions about companies, risks, and financial metrics by querying the database directly.
    
    Use this tool for fact-based questions like:
    - Which company faces the most risk factors?
    - What companies are in the database?
    - How many risk factors does Apple face?
    - What products does NVIDIA mention?
    
    Args:
        query: A natural language question about companies, risks, or financial metrics
    """
    try:
        results = text2cypher_retriever.search(query_text=query)
        if not results.items:
            return "No results found for the query."
        return "\n\n".join(item.content for item in results.items)
    except Exception as e:
        return f"Error querying database: {e}"


# Add the tools to a list
tools = [get_graph_schema, retrieve_financial_documents, query_database]

print(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

## 6. Create Agent

In [ ]:
# Initialize LLM using AWS Bedrock
llm = ChatBedrockConverse(
    model=MODEL_ID,
    provider="anthropic",
    region_name=REGION,
    temperature=0,
)

# Create the agent with tools
SYSTEM_PROMPT = """You are a helpful assistant that can answer questions about 
a graph database containing financial documents. You have three tools:

1. get_graph_schema - Get the database schema
2. retrieve_financial_documents - Search documents semantically
3. query_database - Query specific facts from the database

Choose the appropriate tool based on the question type:
- Use get_graph_schema for questions about database structure
- Use retrieve_financial_documents for semantic/content questions about documents
- Use query_database for specific facts, counts, or entity lookups

When a tool returns data, use that data to answer the question directly.
Be concise and informative in your responses."""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print("Multi-tool agent created!")

## 7. Run the Agent

In [ ]:
def run_agent(question: str):
    """Run the agent with a question and display the response."""
    print(f"User: {question}")
    print("-" * 50)
    
    result = agent.invoke({"messages": [("human", question)]})
    
    # Get the final message
    final_message = result["messages"][-1]
    print(f"\nAssistant: {final_message.content}")
    return result

In [ ]:
# Run the agent with a Text2Cypher question
query = "Which company faces the most risk factors? What are 10 of those risk factors?"
result = run_agent(query)

## 8. Experiment

Depending what question you ask, the agent will use different tools to respond to the question.

---

Modify the question and observe how the agent changes tools, or even runs multiple tools, to gather the context it requires to answer the question.

Try these examples that work well with the database:

**Text2Cypher queries (specific facts):**
* Which company faces the most risk factors?
* What companies are in the database?
* How many risk factors does APPLE INC face?
* What products does NVIDIA mention?
* What stock has MICROSOFT CORP issued?

**Semantic search queries (document content):**
* What are the main risk factors mentioned in the documents?
* What products does Microsoft mention in its financial documents?
* Summarize Apple's business strategy

**Schema queries:**
* How does the graph model relate to financial documents and risk factors?

In [ ]:
# Try: Text2Cypher query
query = "What companies are in the database?"
result = run_agent(query)

In [ ]:
# Try: Semantic search query
query = "What products does Microsoft mention in its financial documents?"
result = run_agent(query)

In [ ]:
# Try: Schema query
query = "How does the graph model relate financial documents to risk factors?"
result = run_agent(query)

In [ ]:
# Try: Another Text2Cypher query
query = "What products does NVIDIA mention?"
result = run_agent(query)

---

## Congratulations!

You have completed Lab 7 - GraphRAG Agents!

You now know how to:
- Build knowledge graphs from unstructured documents
- Implement multiple retrieval strategies (Vector, VectorCypher, Text2Cypher)
- Create intelligent agents that automatically choose the right tool for each question

**You have completed the main workshop!**

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")